# MVSA Dataset to Parquet Exporter
This notebook acts as a standalone script to process the raw MVSA text and images, resize them to 224x224 natively, and store them directly inside a `.parquet` container. All models load this standard artifact.

In [8]:


import os


dataset_path = 'D:/MVSA_SINGLE'
if os.path.exists(dataset_path):
    print(f'Drive mounted. Dataset path: {dataset_path}')
    print(f'Contents: {os.listdir(dataset_path)[:8]}')
else:
    print(f'Path not found: {dataset_path}')


Drive mounted. Dataset path: D:/MVSA_SINGLE
Contents: ['data', 'labelResultAll.txt', 'mvsa_single_processed_224.parquet']


In [9]:
# =====================================================
# Cell 2 - Install Dependencies (optional but recommended)
# =====================================================
%pip install pandas pyarrow pillow -q


Note: you may need to restart the kernel to use updated packages.


In [10]:
# =====================================================
# Cell 3 - Export Parquet Dataset
# =====================================================
import pandas as pd
import os
from PIL import Image
import io
import gc
from concurrent.futures import ThreadPoolExecutor

labels_file = 'D:/MVSA_SINGLE/labelResultAll.txt'
data_dir    = 'D:/MVSA_SINGLE/data'
parquet_path = 'D:/MVSA_SINGLE/mvsa_single_processed_224.parquet'
target_image_size = (224, 224)

def process_single_row(row):
    """Process one image-text pair and store the image at the final training size."""
    try:
        item_id    = str(int(row['id']))
        image_path = os.path.join(data_dir, f"{item_id}.jpg")
        text_path  = os.path.join(data_dir, f"{item_id}.txt")
        t_label    = str(row['text']).strip()
        i_label    = str(row['image']).strip()

        # Determine final sentiment
        if t_label == 'neutral':     f_label = i_label
        elif i_label == 'neutral':   f_label = t_label
        elif t_label == i_label:     f_label = t_label
        else:                        return None

        if os.path.exists(image_path) and os.path.exists(text_path):
            with Image.open(image_path) as img:
                img = img.convert('RGB').resize(target_image_size, Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                img.save(buf, format='PNG')
                img_bytes = buf.getvalue()
            with open(text_path, 'r', encoding='latin-1') as f:
                text_content = f.read().strip()
            return {
                'ID': item_id, 'text_sentiment': t_label,
                'image_sentiment': i_label, 'final_sentiment': f_label,
                'image_bytes': img_bytes, 'text_content': text_content,
            }
    except Exception as e:
        print(f"Error processing row: {e}")
    return None

print('Building from raw data...')
df_labels = pd.read_csv(labels_file, sep=',', header=0)
print(f"Total labels: {len(df_labels)}")
rows = [row for _, row in df_labels.iterrows()]
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_single_row, rows))
processed = [r for r in results if r is not None]
print(f"Processed: {len(processed)} records")
if processed:
    df = pd.DataFrame(processed)
    df.to_parquet(parquet_path, index=False)
    print(f"Saved to: {parquet_path}")
    print(f"Size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")
    del results, processed, df_labels, rows
    gc.collect()
else:
    raise RuntimeError('No data processed! Check paths.')


Building from raw data...
Total labels: 4869
Processed: 4511 records
Saved to: D:/MVSA_SINGLE/mvsa_single_processed_224.parquet
Size: 331.9 MB
